### Dim_RoadSegment

In [0]:
CREATE OR REPLACE TEMP VIEW road_segment_tv AS
SELECT
    road_name,
    FIRST(frc) AS road_category_id,
    FIRST(length_of_road_section) AS length_of_road_segment_km,
    CASE
        WHEN FIRST(frc) = 'FRC0' THEN 'Najvazniji putevi - autoputevi / glavne saobracajnice'
        WHEN FIRST(frc) = 'FRC1' THEN 'Glavni putevi / vazni gradski koridori'
        WHEN FIRST(frc) = 'FRC2' THEN 'Znacajni regionalni / gradski putevi'
        WHEN FIRST(frc) = 'FRC3' THEN 'Ostali vazniji gradski putevi'
        WHEN FIRST(frc) = 'FRC4' THEN 'Lokalni putevi'
        WHEN FIRST(frc) = 'FRC5' THEN 'Manji lokalni putevi'
    END AS road_category,
    FIRST(coordinates) as coordinates
FROM bg_traffic.bg_traffic_silver.tomtom_flow
WHERE road_name IS NOT NULL
GROUP BY road_name;


In [0]:
MERGE INTO bg_traffic.bg_traffic_gold.dim_road_segment AS target
USING road_segment_tv AS source
ON target.road_name = source.road_name
WHEN MATCHED THEN UPDATE SET
    target.road_category_id = source.road_category_id,
    target.length_of_road_segment_km = source.length_of_road_segment_km,
    target.road_category = source.road_category,
    target.coordinates = source.coordinates;

In [0]:
INSERT INTO bg_traffic.bg_traffic_gold.dim_road_segment
SELECT
    (SELECT MAX(road_segment_sid) FROM bg_traffic.bg_traffic_gold.dim_road_segment)
    + ROW_NUMBER() OVER (ORDER BY road_name) AS road_segment_sid,
    road_name,
    road_category_id,
    length_of_road_segment_km,
    road_category,
    coordinates
FROM road_segment_tv
WHERE road_name NOT IN (SELECT road_name FROM bg_traffic.bg_traffic_gold.dim_road_segment);


### Dim_GTFS_Route

In [0]:
CREATE OR REPLACE TEMP VIEW gtfs_route_tv AS
SELECT
    r.route_id,
    r.route_short_name,
    r.route_long_name,
    r.route_type AS route_type_id,
    CASE
        WHEN r.route_type = 0 THEN 'Tramvaj'
        WHEN r.route_type = 3 THEN 'Autobus'
        WHEN r.route_type = 11 THEN 'Trolejbus'
        WHEN r.route_type = 2 THEN 'Železnica'
    END AS route_type_name,
    d.total_distance_km
FROM bg_traffic.bg_traffic_silver.gtfs_route r 
LEFT JOIN(
    SELECT
        t.route_id,
        MAX(s.shape_dist_traveled) AS total_distance_km
    FROM bg_traffic.bg_traffic_silver.gtfs_trip t
    JOIN bg_traffic.bg_traffic_silver.gtfs_shape s ON t.shape_id = s.shape_id
    GROUP BY t.route_id
) d ON r.route_id = d.route_id;

In [0]:
MERGE INTO bg_traffic.bg_traffic_gold.dim_gtfs_route AS target
USING gtfs_route_tv AS source
ON target.route_id = source.route_id
WHEN MATCHED THEN UPDATE SET
    target.route_short_name = source.route_short_name,
    target.route_long_name = source.route_long_name,
    target.route_type_id = source.route_type_id,
    target.route_type_name = source.route_type_name,
    target.total_distance_km = source.total_distance_km;

In [0]:
INSERT INTO bg_traffic.bg_traffic_gold.dim_gtfs_route
SELECT
    (SELECT MAX(route_sid) FROM bg_traffic.bg_traffic_gold.dim_gtfs_route)
    + ROW_NUMBER() OVER (ORDER BY route_id) AS route_sid,
    route_id,
    route_short_name,
    route_long_name,
    route_type_id,
    route_type_name,
    total_distance_km
FROM gtfs_route_tv
WHERE route_id NOT IN (SELECT route_id FROM bg_traffic.bg_traffic_gold.dim_gtfs_route);

### Dim_GTFS_Stop

In [0]:
CREATE OR REPLACE TEMP VIEW gtfs_stop_tv AS
SELECT
    stop_id,
    stop_code,
    stop_name,
    stop_latitude,
    stop_longitude
FROM bg_traffic.bg_traffic_silver.gtfs_stop;

In [0]:
MERGE INTO bg_traffic.bg_traffic_gold.dim_gtfs_stop AS target
USING gtfs_stop_tv AS source
ON target.stop_id = source.stop_id
WHEN MATCHED THEN UPDATE SET
    target.stop_code = source.stop_code,
    target.stop_name = source.stop_name,
    target.stop_latitude = source.stop_latitude,
    target.stop_longitude = source.stop_longitude;


In [0]:
INSERT INTO bg_traffic.bg_traffic_gold.dim_gtfs_stop
SELECT
    (SELECT MAX(stop_sid) FROM bg_traffic.bg_traffic_gold.dim_gtfs_stop)
    + ROW_NUMBER() OVER (ORDER BY stop_id) AS stop_sid,
    stop_id,
    stop_code,
    stop_name,
    stop_latitude,
    stop_longitude
FROM gtfs_stop_tv
WHERE stop_id NOT IN (SELECT stop_id FROM bg_traffic.bg_traffic_gold.dim_gtfs_stop);


### Dim_AirStation

In [0]:
CREATE OR REPLACE TEMP VIEW air_station_tv AS
SELECT DISTINCT
    location_id,
    location_name,
    latitude,
    longitude,
    provider_name
FROM bg_traffic.bg_traffic_silver.openaq_locations;

In [0]:
MERGE INTO bg_traffic.bg_traffic_gold.dim_air_station AS target
USING air_station_tv AS source
ON target.location_id = source.location_id
WHEN MATCHED THEN UPDATE SET
    target.location_name = source.location_name,
    target.latitude = source.latitude,
    target.longitude = source.longitude,
    target.provider_name = source.provider_name;

In [0]:
INSERT INTO bg_traffic.bg_traffic_gold.dim_air_station
SELECT
    (SELECT MAX(location_sid) FROM bg_traffic.bg_traffic_gold.dim_air_station)
    + ROW_NUMBER() OVER (ORDER BY location_id) AS location_sid,
    location_id,
    location_name,
    latitude,
    longitude,
    provider_name
FROM air_station_tv
WHERE location_id NOT IN (SELECT location_id FROM bg_traffic.bg_traffic_gold.dim_air_station);